<a href="https://colab.research.google.com/github/mritunjay-chandra-jha/operation_research_problems/blob/main/mixed_integer_problem_production_planning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Production Planning with Setup Costs
### Building a mixed-integer model in Gurobi (continuous, binary and integer variables)

**Deterministic Decision Modeling Lab**

---
### What you will learn
1. When to use **continuous**, **binary** and **integer** variables
2. How to store data in Python **dictionaries**
3. How to create variables and constraints for *many items at once* with `addVars` and `addConstrs`
4. How to solve the model, check the result, and test "what-if" changes

### How to use this notebook
- Read each **text cell** first, then run the **code cell** below it (click the cell and press **Shift + Enter**).
- Run the cells **from top to bottom, in order**. If you get a `NameError`, you skipped a cell: use *Runtime → Restart and run all*.
- Do not just run the code. Try to predict what each cell will print before you run it.

---
# Part 1: The Problem

A small factory can make three products: **A, B and C**.

- Every unit sold earns a **profit**.
- To make a product at all, the factory must first pay a one-time **setup cost** (preparing the machine, buying tooling, and so on). If a product is **not made, its setup cost is not paid**.
- Every unit uses some **labor hours**, and the factory has only **120 labor hours** in total.
- Customers will buy only a limited number of units of each product, the **market limit**. There is no point producing more than that.
- The machines work in **batches**. One batch produces exactly **10 units**, and each product can run at most **5 batches**. So the quantity of each product must be a multiple of 10.

> **Question:** Which products should the factory make, and how many units of each, to earn the highest **net profit**?
>
> Net profit = (profit from units sold) − (setup costs paid)

### Data

| | Profit per unit | Setup cost | Labor hours per unit | Market limit (units) |
|---|:---:|:---:|:---:|:---:|
| **Product A** | 40 | 200 | 2 | 40 |
| **Product B** | 30 | 150 | 1 | 60 |
| **Product C** | 50 | 300 | 3 | 30 |

**Other data:** total labor available = **120 hours**, batch size = **10 units**, maximum batches per product = **5**.

**Before you continue, try to guess:** which products do you think the factory will make? Write your guess down and compare it with the answer at the end.

---
# Part 2: Thinking Before Modeling

Every optimization model answers three questions. Answer them in this order, on paper, **before** you write any code.

| Question | For this problem |
|---|---|
| **1. What can we decide?** (decision variables) | How many units of each product? Do we make each product at all? How many batches? |
| **2. What do we want?** (objective) | Maximize net profit |
| **3. What limits us?** (constraints) | Labor hours, market limits, batch rule, and "no setup, no production" |

### Choosing the type of each variable
The type of a variable tells the solver what values it is allowed to take. This is the most important modeling choice in this lab.

| Decision | Possible values | Variable type | Gurobi name |
|---|---|---|---|
| Units of each product to make | Any number 0 or more | **Continuous** | `GRB.CONTINUOUS` |
| Do we make the product at all? (pay the setup cost?) | Only **yes (1)** or **no (0)** | **Binary** | `GRB.BINARY` |
| Number of batches | Whole numbers 0, 1, 2, ..., 5 | **Integer** | `GRB.INTEGER` |

**Why do we need a yes/no variable?** The setup cost is paid *once*, no matter how many units we make, and only if we make *any*. A yes/no variable is the only way to switch that cost on and off.

---
# Part 3: The Mathematical Model

### 3.1 Sets and data (these are numbers we are given)

| Symbol | Meaning | Values |
|---|---|---|
| $P$ | Set of products | $\{A, B, C\}$ |
| $\pi_p$ | Profit per unit of product $p$ | A: 40, B: 30, C: 50 |
| $f_p$ | Setup cost of product $p$ | A: 200, B: 150, C: 300 |
| $h_p$ | Labor hours per unit of product $p$ | A: 2, B: 1, C: 3 |
| $u_p$ | Market limit (maximum units) of product $p$ | A: 40, B: 60, C: 30 |
| $L$ | Total labor hours available | 120 |
| $B$ | Units in one batch | 10 |
| $K$ | Maximum batches per product | 5 |

### 3.2 Decision variables (these are what the solver chooses)

| Symbol | Meaning | Type and range |
|---|---|---|
| $x_p$ | Number of **units** of product $p$ to make | Continuous, $x_p \ge 0$ |
| $y_p$ | **1** if product $p$ is made (setup cost paid), **0** if not | Binary, $y_p \in \{0, 1\}$ |
| $k_p$ | Number of **batches** of product $p$ | Integer, $k_p \in \{0, 1, \dots, K\}$ |

### 3.3 Objective: maximize net profit
Profit from the units we sell, minus the setup costs of the products we decide to make:

$$\max \;\; \sum_{p \in P} \left( \pi_p \, x_p \;-\; f_p \, y_p \right)$$

*"for each product, take profit per unit times units, then subtract the setup cost times the yes/no variable; add this up over all products."*
If $y_p = 0$, the setup cost $f_p y_p$ is $0$ and we pay nothing for that product.

### 3.4 Constraints

**(1) Link: no setup, no production. Never exceed the market limit.**

$$x_p \le u_p \, y_p \qquad \text{for every } p \in P$$

- If $y_p = 0$ the right-hand side is $0$, so $x_p$ must be $0$: *we cannot make a product we did not set up.*
- If $y_p = 1$ the right-hand side is $u_p$, so $x_p$ can be anything up to the market limit.

**(2) Labor capacity.**

$$\sum_{p \in P} h_p \, x_p \le L$$

The total labor hours used by all products cannot exceed 120.

**(3) Batch rule: production is a whole number of batches.**

$$x_p = B \, k_p \qquad \text{for every } p \in P$$

Because $k_p$ is an integer, $x_p$ can only be 0, 10, 20, 30, 40 or 50.

**(4) Variable domains.**

$$x_p \ge 0, \qquad y_p \in \{0,1\}, \qquad k_p \in \{0,1,\dots,K\} \qquad \text{for every } p \in P$$

### 3.5 The same model with the numbers filled in

$$\max \;\; 40x_A + 30x_B + 50x_C - 200y_A - 150y_B - 300y_C$$

$$x_A \le 40\,y_A, \qquad x_B \le 60\,y_B, \qquad x_C \le 30\,y_C$$

$$2x_A + 1x_B + 3x_C \le 120$$

$$x_A = 10\,k_A, \qquad x_B = 10\,k_B, \qquad x_C = 10\,k_C$$

$$x \ge 0, \qquad y \in \{0,1\}, \qquad k \in \{0,1,2,3,4,5\}$$

This is a **mixed-integer linear program (MILP)**: every formula is linear, but some variables must be whole numbers or yes/no. Gurobi solves this kind of model with a method called *branch and bound*.

---
# Part 4: From Math to Code

Here is how every symbol in the model will appear in our Python code. Come back to this table whenever you are unsure what a line means.

| In the math | In the code | How it is created |
|---|---|---|
| Set $P$ | `products` | A Python **list**: `["A", "B", "C"]` |
| $\pi_p$, $f_p$, $h_p$, $u_p$ | `profit`, `setup`, `hours`, `maxunits` | Python **dictionaries** |
| $L$, $B$, $K$ | `labor_available`, `batch_size`, `max_batches` | Plain numbers |
| $x_p$ (continuous) | `x` | `m.addVars(products, ..., vtype=GRB.CONTINUOUS)` |
| $y_p$ (binary) | `y` | `m.addVars(products, ..., vtype=GRB.BINARY)` |
| $k_p$ (integer) | `k` | `m.addVars(products, ..., vtype=GRB.INTEGER)` |
| Objective | `m.setObjective(...)` | with `gp.quicksum(...)` for the sum |
| Constraint (1) | `link` | `m.addConstrs(...)` (many constraints at once) |
| Constraint (2) | `labor` | `m.addConstr(...)` (one constraint) |
| Constraint (3) | `batch` | `m.addConstrs(...)` (many constraints at once) |

### The Gurobi workflow
Every Gurobi program has the same steps. We will follow them in order:

1. **Install and import** the library
2. **Enter the data**
3. **Create** an empty model
4. **Add variables**
5. **Set the objective**
6. **Add constraints**
7. **Check** the model
8. **Optimize**
9. **Read and verify** the results

---
# Part 5: Building the Model in Gurobi

## Step 1: Install and import Gurobi

Google Colab does not come with Gurobi installed, so we install it with `pip`, Python's package installer.

- The `!` at the start tells Colab to run the line as a **terminal command**, not as Python.
- `-q` means *quiet*, so we do not get a long wall of text.
- The version installed with `pip` includes a **free size-limited license** (up to 2000 variables and 2000 constraints). Our model has only 9 variables and 7 constraints, so this is more than enough.

You need to run this cell **every time you open a new Colab session**, because Colab resets its machine.

In [ ]:
!pip install gurobipy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 106.4 MB/s eta 0:00:00


Now we **import** the library into Python.

- `import gurobipy as gp` loads Gurobi and gives it the short nickname `gp`. Every Gurobi function we use will start with `gp.`
- `from gurobipy import GRB` loads a bundle of **constants** (fixed names) such as `GRB.MAXIMIZE`, `GRB.BINARY` and `GRB.OPTIMAL`, so we can write `GRB.BINARY` instead of `gp.GRB.BINARY`.
- The last line prints the installed version. It is a quick check that the installation worked. Your version number may differ from your classmates', and that is fine.

In [ ]:
import gurobipy as gp
from gurobipy import GRB

print("Gurobi version:", gp.gurobi.version())

Gurobi version: (13, 0, 3)


## Step 2: Enter the data

We keep the **data separate from the model**. If a number changes tomorrow (for example, the profit of B becomes 35), we edit one value in the data cell and the rest of the code stays the same.

### Python lists and dictionaries
- A **list** holds items in order: `products = ["A", "B", "C"]`. We use it for the **set of products**, $P$.
- A **dictionary** connects a **key** to a **value**: `profit = {"A": 40, "B": 30, "C": 50}`.
  Here the keys are the product names and the values are the profits. Writing `profit["B"]` gives `30`.

Dictionaries are ideal for optimization data because every number is tied to a **name**. You always know that `40` belongs to product `"A"`.

| Code name | Math symbol | Meaning |
|---|---|---|
| `products` | $P$ | Names of the products |
| `profit` | $\pi_p$ | Profit per unit |
| `setup` | $f_p$ | Setup cost (paid once, only if the product is made) |
| `hours` | $h_p$ | Labor hours per unit |
| `maxunits` | $u_p$ | Market limit in units |
| `labor_available` | $L$ | Total labor hours (a single number) |
| `batch_size` | $B$ | Units in one batch (a single number) |
| `max_batches` | $K$ | Maximum batches per product (a single number) |

Anything after a `#` is a **comment**: a note for humans that Python ignores.

In [ ]:
# Names of the products (the set P)
products = ["A", "B", "C"]

# Data that depends on the product: dictionaries {product: value}
profit   = {"A": 40,  "B": 30,  "C": 50}     # profit per unit sold
setup    = {"A": 200, "B": 150, "C": 300}    # one-time setup cost
hours    = {"A": 2,   "B": 1,   "C": 3}      # labor hours used per unit
maxunits = {"A": 40,  "B": 60,  "C": 30}     # market limit (units)

# Data that does not depend on the product: single numbers
labor_available = 120     # total labor hours
batch_size      = 10      # units produced in one batch
max_batches     = 5       # maximum batches per product

### Check your data
Always look at your data before building the model. A typo here would silently give a wrong answer later.

The cell below prints the dictionaries as a small table using **pandas**, a Python library for tables. You do not need to understand the pandas part today. Just check that the table matches the data table in Part 1.

In [ ]:
import pandas as pd

pd.DataFrame({"profit": profit, "setup": setup, "hours": hours, "maxunits": maxunits})

,profit,setup,hours,maxunits
A,40,200,2,40
B,30,150,1,60
C,50,300,3,30


**Try it:** in a new code cell, type `profit["B"]` and run it. You should see `30`. Then try `hours["C"]`. This is exactly how the model will look up numbers later.

## Step 3: Create an empty model

`gp.Model("name")` creates a new, empty optimization model and stores it in the Python variable `m`. Think of `m` as a blank notebook where we will write our variables, objective and constraints. The text in quotes is just a label that appears in the solver output.

In [ ]:
m = gp.Model("production_plan")

## Step 4: Add the decision variables

We use `m.addVars(...)`, with an **s** at the end. It creates **one variable for every item in a list, all in one line**. Since our list is `products = ["A", "B", "C"]`, one call creates three variables.

The general form is:

```python
m.addVars(index_list, lb=lower_bound, ub=upper_bound, vtype=type, name="label")
```

| Part | Meaning |
|---|---|
| `index_list` | One variable is created for each item in this list |
| `lb` | **L**ower **b**ound: the smallest value allowed (default is 0) |
| `ub` | **U**pper **b**ound: the largest value allowed (default is infinity) |
| `vtype` | Variable type: `GRB.CONTINUOUS`, `GRB.BINARY` or `GRB.INTEGER` |
| `name` | A label, so variables appear as `x[A]`, `x[B]`, ... in outputs |

### Variable 1: units produced, $x_p$ (continuous)
How many units of each product do we make? Any number that is 0 or more, so the type is **continuous** and the lower bound is 0.

After this line we can refer to `x["A"]`, `x["B"]` and `x["C"]`.

In [ ]:
x = m.addVars(products, lb=0, vtype=GRB.CONTINUOUS, name="x")


### Variable 2: make the product or not, $y_p$ (binary)
Do we make the product at all? Only two answers are possible: **yes (1)** or **no (0)**. This is a **binary** variable.

For binary variables Gurobi automatically sets the bounds to 0 and 1, so we do not need `lb` or `ub`. This variable is what switches the setup cost on and off:

- `y["A"] = 1` means *"we make product A, so we pay its setup cost of 200"*
- `y["A"] = 0` means *"we do not make A, and we pay nothing"*

In [ ]:
y = m.addVars(products, vtype=GRB.BINARY, name="y")


### Variable 3: number of batches, $k_p$ (integer)
How many batches of each product do we run? Only whole numbers make sense (you cannot run 2.5 batches), and at most 5. So this is an **integer** variable with `lb=0` and `ub=max_batches`.

Notice that we use the **name** `max_batches` that we defined in the data cell, not the number 5. If the limit changes, we change it in one place only.

In [ ]:
k = m.addVars(products, lb=0, ub=max_batches, vtype=GRB.INTEGER, name="k")

### Quick check
`m.update()` tells Gurobi to register everything we added so far (Gurobi adds things "lazily"). Then we can count what the model contains. You should see **9 variables**: 3 continuous + 3 binary + 3 integer.

(Gurobi counts binary variables as a special kind of integer variable, so `NumIntVars` shows 6 = 3 integer + 3 binary.)

In [ ]:
m.update()
print("Variables:", m.NumVars)
print("  of which binary :", m.NumBinVars)
print("  of which integer (including binary):", m.NumIntVars)

Variables: 9
  of which binary : 3
  of which integer (including binary): 6


## Step 5: Set the objective

The objective is to maximize

$$\sum_{p \in P} \left( \pi_p \, x_p - f_p \, y_p \right)$$

In code, `profit[p]*x[p] - setup[p]*y[p]` is the term for **one** product `p`. To add it up over all products we use `gp.quicksum(...)`:

```python
gp.quicksum( term  for p in products )
```

Read `for p in products` as the Σ symbol: *"for each product p"*. Python runs through "A", "B", "C" in turn, builds each term, and adds them all up.

We use `gp.quicksum` (Gurobi's own fast sum) rather than Python's built-in `sum` when we are adding **variables**.

Finally, `GRB.MAXIMIZE` tells Gurobi we want the largest value. Use `GRB.MINIMIZE` for a cost-minimization model.

> **Note:** `40*x["A"] + ...` does not calculate a number. Gurobi stores it as a **formula** to be optimized later. Nothing is solved until we call `m.optimize()`.

In [ ]:
m.setObjective(gp.quicksum(profit[p]*x[p] - setup[p]*y[p] for p in products), GRB.MAXIMIZE)

## Step 6: Add the constraints

Two functions add constraints:

| Function | Use it when | Example |
|---|---|---|
| `m.addConstr(...)` (singular) | You have **one** constraint | Total labor limit |
| `m.addConstrs(...)` (plural, with an **s**) | You need **one constraint per item**, many at once | One link constraint for each product |

For `addConstrs` we write the constraint followed by `for p in products`, wrapped in an extra pair of parentheses:

```python
m.addConstrs(( constraint  for p in products ), name="label")
```

Always give each constraint a `name`. It makes the model easy to read and debug.

### Constraint (1): the link, "no setup, no production"

$$x_p \le u_p \, y_p \qquad \text{for every } p \in P$$

This is the most important constraint of the model. Think about the two cases for product A (market limit 40):

| If... | Then the constraint says | Meaning |
|---|---|---|
| `y["A"] = 0` | `x["A"] <= 40 * 0 = 0` | We did not set up A, so we **cannot make any** |
| `y["A"] = 1` | `x["A"] <= 40 * 1 = 40` | Setup paid, so we can make **up to the market limit** |

This single line creates three constraints: `link[A]`, `link[B]` and `link[C]`.

In [ ]:
m.addConstrs((x[p] <= maxunits[p]*y[p] for p in products), name="link")

{'A': <gurobi.Constr *Awaiting Model Update*>,
 'B': <gurobi.Constr *Awaiting Model Update*>,
 'C': <gurobi.Constr *Awaiting Model Update*>}

### Constraint (2): labor capacity

$$\sum_{p \in P} h_p \, x_p \le L$$

All the products compete for the same 120 labor hours. The hours used are
`hours["A"]*x["A"] + hours["B"]*x["B"] + hours["C"]*x["C"]`, which is `gp.quicksum(hours[p]*x[p] for p in products)`.

This is **one** constraint that adds up all the products, so we use the singular `addConstr`. There is no `for p in products` outside the sum.

In [ ]:
m.addConstr(gp.quicksum(hours[p]*x[p] for p in products) <= labor_available, name="labor")

<gurobi.Constr *Awaiting Model Update*>

### Constraint (3): the batch rule

$$x_p = B \, k_p \qquad \text{for every } p \in P$$

Units made must equal the batch size times the number of batches. Because `k[p]` is an integer, `x[p]` can only be 0, 10, 20, 30, 40 or 50.

Look carefully at the symbol: in a constraint, **equality is written with a double equals sign `==`**. A single `=` is for assigning a value to a Python variable, and will give an error here.

In [ ]:
m.addConstrs((x[p] == batch_size*k[p] for p in products), name="batch")

{'A': <gurobi.Constr *Awaiting Model Update*>,
 'B': <gurobi.Constr *Awaiting Model Update*>,
 'C': <gurobi.Constr *Awaiting Model Update*>}

## Step 7: Look at the model before solving

A very good habit is to **read your model** before asking Gurobi to solve it. A mistake is much easier to find now than after you get a strange answer.

- `m.update()` registers the latest changes.
- `m.write("model.lp")` saves the model as a text file in a readable format, called an *LP file*.
- `print(open("model.lp").read())` opens that file and prints it here.

Compare the output with the "numbers filled in" model in Part 3.5. Check these things:
1. Does the objective match? (`40 x[A] + 30 x[B] + 50 x[C] - 200 y[A] ...`)
2. Are there three `link` constraints, one `labor` constraint and three `batch` constraints? (7 in total)
3. Are `y` listed under **Binaries** and `k` under **Generals** (integers)?

In [ ]:
m.update()
m.write("model.lp")
print(open("model.lp").read())

\ Model production_plan
\ LP format - for model browsing. Use MPS format to capture full model detail.
Maximize
  40 x[A] + 30 x[B] + 50 x[C] - 200 y[A] - 150 y[B] - 300 y[C]
Subject To
 link[A]: x[A] - 40 y[A] <= 0
 link[B]: x[B] - 60 y[B] <= 0
 link[C]: x[C] - 30 y[C] <= 0
 labor: 2 x[A] + x[B] + 3 x[C] <= 120
 batch[A]: x[A] - 10 k[A] = 0
 batch[B]: x[B] - 10 k[B] = 0
 batch[C]: x[C] - 10 k[C] = 0
Bounds
 k[A] <= 5
 k[B] <= 5
 k[C] <= 5
Binaries
 y[A] y[B] y[C]
Generals
 k[A] k[B] k[C]
End



## Step 8: Solve the model

`m.optimize()` is the one line that runs the solver. Everything before it only **described** the problem. Now Gurobi searches for the best plan.

Because our model has integer and binary variables, Gurobi uses **branch and bound**: it first solves an easier version where whole numbers are not required, then splits ("branches") the problem step by step until every variable has an allowed value, while proving that nothing better exists.

### How to read the output
You do not need to read every line. Look at these parts:

| Part of the log | What it tells you |
|---|---|
| `Optimize a model with 7 rows, 9 columns` | Gurobi saw **7 constraints** and **9 variables**, as we expected |
| `Variable types: 3 continuous, 6 integer (3 binary)` | The variable types were read correctly |
| `Found heuristic solution` / the lines starting with `H` | Gurobi found feasible plans along the way, each better than the last |
| `Incumbent` | The **best plan found so far** |
| `BestBd` | The best value that is **mathematically possible** (an upper bound for a maximization problem) |
| `Gap` | The difference between the two. When it reaches **0%**, the incumbent is proven optimal |
| `Optimal solution found` | Success |

In [ ]:
m.optimize()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.5 LTS")

CPU model: AMD EPYC 7B12, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 7 rows, 9 columns and 15 nonzeros (Max)
Model fingerprint: 0x90cfe33c
Model has 6 linear objective coefficients
Variable types: 3 continuous, 6 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+01]
  Objective range  [3e+01, 3e+02]
  Bounds range     [1e+00, 5e+00]
  RHS range        [1e+02, 1e+02]

Found heuristic solution: objective -0.0000000
Presolve removed 3 rows and 3 columns
Presolve time: 0.00s
Presolved: 4 rows, 6 columns, 9 nonzeros
Variable types: 0 continuous, 6 integer (3 binary)

Root relaxation: objective 2.575000e+03, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time


## Step 9: Read the results

### First, check the status
After `m.optimize()`, the number `m.Status` tells us how the solve ended. **Always check it before reading any numbers.** If the model is infeasible or the solver ran out of time, the variable values are not a valid answer.

| Status | Meaning |
|---|---|
| `GRB.OPTIMAL` | Solved. Proven best solution found |
| `GRB.INFEASIBLE` | No plan satisfies all the constraints |
| `GRB.UNBOUNDED` | The objective can improve forever (usually a missing constraint) |
| `GRB.TIME_LIMIT` | Stopped by a time limit |

### Reading values
- `m.ObjVal` is the optimal objective value (the maximum net profit).
- `x[p].X` is the value of variable `x[p]` in the solution. **The `X` must be a capital letter.**
- For a binary variable we test `y[p].X > 0.5` instead of `== 1`. The solver may return `0.9999999` because of tiny rounding, so comparing with 0.5 is safer.
- `round(...)` removes this tiny noise (for example, it avoids printing `-0`).

In [ ]:
if m.Status == GRB.OPTIMAL:
    print(f"Maximum net profit: {m.ObjVal:.2f}\n")
    for p in products:
        print(f"Product {p}:  make it? {y[p].X > 0.5}   units = {round(x[p].X)}   batches = {round(k[p].X)}")
else:
    print("The model was not solved to optimality. Status code:", m.Status)

Maximum net profit: 2450.00

Product A:  make it? True   units = 40   batches = 4
Product B:  make it? True   units = 40   batches = 4
Product C:  make it? False   units = 0   batches = 0


### The solution as a table
We build one row per product with a `for` loop and show it as a table. `y[p].X > 0.5` becomes `"Yes"` or `"No"`.

The column **Profit contribution** is units × profit minus the setup cost, so the products' contributions add up to the net profit.

In [ ]:
rows = []
for p in products:
    made  = y[p].X > 0.5
    units = round(x[p].X)
    rows.append({
        "Product": p,
        "Made?": "Yes" if made else "No",
        "Batches": round(k[p].X),
        "Units": units,
        "Labor used": hours[p] * units,
        "Profit contribution": profit[p]*units - (setup[p] if made else 0),
    })

results = pd.DataFrame(rows)
results.style.hide(axis="index")

Product,Made?,Batches,Units,Labor used,Profit contribution
A,Yes,4,40,80,1400
B,Yes,4,40,40,1050
C,No,0,0,0,0


## Step 10: Verify the answer by hand

Never trust a result you have not checked. Let us rebuild the answer from the numbers, without using `m.ObjVal`, and make sure every constraint is satisfied.

In [ ]:
units = {p: round(x[p].X) for p in products}
made  = {p: y[p].X > 0.5 for p in products}

revenue    = sum(profit[p] * units[p] for p in products)
setup_paid = sum(setup[p] for p in products if made[p])
labor_used = sum(hours[p] * units[p] for p in products)

print(f"Profit from units sold : {revenue}")
print(f"Setup costs paid       : {setup_paid}")
print(f"Net profit             : {revenue - setup_paid}   (Gurobi says {m.ObjVal:.0f})")
print()
print(f"Labor used: {labor_used} of {labor_available} hours  ->  within limit? {labor_used <= labor_available}")
for p in products:
    print(f"Product {p}: units {units[p]} <= market limit {maxunits[p]}?  {units[p] <= maxunits[p]}   |   multiple of {batch_size}?  {units[p] % batch_size == 0}")

Profit from units sold : 2800
Setup costs paid       : 350
Net profit             : 2450   (Gurobi says 2450)

Labor used: 120 of 120 hours  ->  within limit? True
Product A: units 40 <= market limit 40?  True   |   multiple of 10?  True
Product B: units 40 <= market limit 60?  True   |   multiple of 10?  True
Product C: units 0 <= market limit 30?  True   |   multiple of 10?  True


### Why this plan?
Look at the answer and think about these questions:

1. **Which products were made, and was your guess in Part 1 correct?**
2. **Why is product C not made?** Labor is the scarce resource: all hours are used. So what matters is the **profit per labor hour** of each product.

   | Product | Profit per unit | Hours per unit | Profit per labor hour |
   |---|---|---|---|
   | A | 40 | 2 | 20.0 |
   | B | 30 | 1 | 30.0 |
   | C | 50 | 3 | 16.7 |

   C has the highest profit per unit, but it uses a lot of labor and has the largest setup cost, so it gives the least profit for each hour of labor.
3. **What would happen if the setup cost did not exist?** (Test it in the exercises below.)

---
# Part 6: Changing the Model

A model is a tool for answering **"what if?"** questions. Gurobi lets us change a model that has already been built and solve it again.

### Adding a new constraint
Suppose management decides that **at most one product** may be set up. In math:

$$\sum_{p \in P} y_p \le 1$$

`y.sum()` adds up all the `y` variables, so the constraint is a single line. We give it a name so we can remove it later.

In [ ]:
m.addConstr(y.sum() <= 1, name="at_most_one_product")
m.optimize()

print(f"\nNet profit with the new rule: {m.ObjVal:.0f}")
for p in products:
    print(f"  Product {p}: units = {round(x[p].X)}")

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.5 LTS")

CPU model: AMD EPYC 7B12, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 8 rows, 9 columns and 18 nonzeros (Max)
Model fingerprint: 0xf2643232
Model has 6 linear objective coefficients
Variable types: 3 continuous, 6 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+01]
  Objective range  [3e+01, 3e+02]
  Bounds range     [1e+00, 5e+00]
  RHS range        [1e+00, 1e+02]

MIP start from previous solve did not produce a new incumbent solution
MIP start from previous solve violates constraint at_most_one_product by 1.000000000

Found heuristic solution: objective -0.0000000
Presolve removed 3 rows and 3 columns
Presolve time: 0.00s
Presolved: 5 rows, 6 columns, 12 nonzeros
Variable types: 0 continuous, 6 integer (3 binary)

Root relaxation: objective 1.400000e+03, 2 iterations, 0.00 seconds (0.00 work 

Adding a constraint can **never improve** the objective. Compare this profit with the earlier one. Why is it lower?

### Removing a constraint
To remove a constraint, find it by its **name** with `getConstrByName` and pass it to `m.remove`. This is why we named every constraint.

In [ ]:
m.remove(m.getConstrByName("at_most_one_product"))
m.optimize()

print(f"\nNet profit after removing the rule: {m.ObjVal:.0f}")
print("Constraints now in the model:", [c.ConstrName for c in m.getConstrs()])

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.5 LTS")

CPU model: AMD EPYC 7B12, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 7 rows, 9 columns and 15 nonzeros (Max)
Model fingerprint: 0x90cfe33c
Model has 6 linear objective coefficients
Variable types: 3 continuous, 6 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+01]
  Objective range  [3e+01, 3e+02]
  Bounds range     [1e+00, 5e+00]
  RHS range        [1e+02, 1e+02]

MIP start from previous solve produced solution with objective 1400 (0.01s)
Loaded MIP start from previous solve with objective 1400

Presolve removed 3 rows and 3 columns
Presolve time: 0.00s
Presolved: 4 rows, 6 columns, 9 nonzeros
Variable types: 0 continuous, 6 integer (3 binary)

Root relaxation: objective 2.575000e+03, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |

The profit returns to its original value, and the list shows that the model has its original 7 constraints again.

### Changing a number in the model
We can also change an existing constraint directly. Here we give the factory **200 labor hours** instead of 120, by changing the right-hand side (`RHS`) of the `labor` constraint.

In [ ]:
m.getConstrByName("labor").RHS = 200
m.optimize()

print(f"\nNet profit with 200 labor hours: {m.ObjVal:.0f}")
for p in products:
    print(f"  Product {p}: make it? {y[p].X > 0.5}   units = {round(x[p].X)}")

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.5 LTS")

CPU model: AMD EPYC 7B12, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 7 rows, 9 columns and 15 nonzeros (Max)
Model fingerprint: 0x6db10705
Model has 6 linear objective coefficients
Variable types: 3 continuous, 6 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+01]
  Objective range  [3e+01, 3e+02]
  Bounds range     [1e+00, 5e+00]
  RHS range        [2e+02, 2e+02]

MIP start from previous solve produced solution with objective 2450 (0.01s)
Loaded MIP start from previous solve with objective 2450

Presolve removed 3 rows and 3 columns
Presolve time: 0.00s
Presolved: 4 rows, 6 columns, 9 nonzeros
Variable types: 0 continuous, 6 integer (3 binary)
Found heuristic solution: objective 2750.0000000

Root relaxation: objective 3.600000e+03, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |

With more labor, the plan changes. Which products are made now, and why?

Now put the labor back to 120 so the notebook is in its original state:

In [ ]:
m.getConstrByName("labor").RHS = labor_available
m.optimize()
print(f"Net profit with {labor_available} labor hours: {m.ObjVal:.0f}")

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.5 LTS")

CPU model: AMD EPYC 7B12, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 7 rows, 9 columns and 15 nonzeros (Max)
Model fingerprint: 0x90cfe33c
Model has 6 linear objective coefficients
Variable types: 3 continuous, 6 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+01]
  Objective range  [3e+01, 3e+02]
  Bounds range     [1e+00, 5e+00]
  RHS range        [1e+02, 1e+02]

MIP start from previous solve did not produce a new incumbent solution

Found heuristic solution: objective -0.0000000
Presolve removed 3 rows and 3 columns
Presolve time: 0.00s
Presolved: 4 rows, 6 columns, 9 nonzeros
Variable types: 0 continuous, 6 integer (3 binary)

Root relaxation: objective 2.575000e+03, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl

---
# Part 7: Common Mistakes and How to Fix Them

| What you see | Likely cause | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'gurobipy'` | You did not run the install cell | Run Step 1, then run the import cell again |
| `NameError: name 'm' is not defined` (or `x`, `profit`, ...) | A cell was skipped or run out of order | *Runtime → Restart and run all* |
| `AttributeError: ... 'x'` when reading a value | You wrote `x[p].x` | The value attribute is a **capital X**: `x[p].X` |
| Output looks like `<gurobi.Var x[A]>` instead of a number | You printed the variable, not its value | Use `x[p].X` |
| `KeyError: 'a'` | The dictionary key does not match exactly | Keys are case-sensitive: `"A"` is not `"a"` |
| `SyntaxError` in a constraint | You used `=` instead of `==`, `<=` or `>=` | A constraint needs a comparison operator |
| Error reading `.X` before solving | You did not call `m.optimize()`, or the model was not optimal | Solve first, and check `m.Status == GRB.OPTIMAL` |
| The profit is not what you expected | A typo in the data, or a missing constraint | Print the data (Step 2) and read the LP file (Step 7) |

---
# Part 8: Practice Exercises

Copy the cells you need into new code cells, change **only the data**, and re-run from the data cell downward (*Runtime → Run after*). Write down what you see **and why**.

**Exercise 1: The setup cost of C.**
Change the setup cost of product C from 300 to 50 in the data cell. Is C made now? Explain the result using profit per labor hour.

**Exercise 2: No setup costs.**
Set all three setup costs to 0. What is the new plan? How does it compare with the plan with setup costs?

**Exercise 3: Smaller batches.**
Set `batch_size = 1` and `max_batches = 60`. The factory can now produce any whole number of units. Does the net profit go up, down or stay the same? Why?

**Exercise 4: A new rule.**
Add a constraint so that the factory makes **at least 50 units of product B**, and solve. What is the new plan and net profit? Why did it get worse?
*Hint:* a constraint on one variable looks like `m.addConstr(x["B"] >= 50, name="min_B")`.

**Exercise 5: Your own product.**
Add a **fourth product D** by editing only the data cell: `products`, `profit`, `setup`, `hours` and `maxunits` each need one new entry. Notice that you do **not** need to change any model code. Why is that?

**Exercise 6: Counting constraints.**
After you add product D, print `m.NumVars` and `m.NumConstrs`. Predict the numbers *before* you run the cell.

Exercise 1: The setup cost of C. Change the setup cost of product C from 300 to 50 in the data cell. Is C made now? Explain the result using profit per labor hour.


In [ ]:
setup = {"A": 200, "B": 150, "C": 50}

In [ ]:
m.update()
m.setObjective(gp.quicksum(profit[p]*x[p] - setup[p]*y[p] for p in products), GRB.MAXIMIZE)
m.optimize()

if m.Status == GRB.OPTIMAL:
    print(f"Maximum net profit: {m.ObjVal:.2f}\n")
    for p in products:
        print(f"Product {p}:  make it? {y[p].X > 0.5}   units = {round(x[p].X)}   batches = {round(k[p].X)}")
else:
    print("The model was not solved to optimality. Status code:", m.Status)

print(f"\nNet profit with 50 setup cost of C: {m.ObjVal:.0f}")

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.5 LTS")

CPU model: AMD EPYC 7B12, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 7 rows, 9 columns and 15 nonzeros (Max)
Model fingerprint: 0xfe471b7d
Model has 6 linear objective coefficients
Variable types: 3 continuous, 6 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+01]
  Objective range  [3e+01, 2e+02]
  Bounds range     [1e+00, 5e+00]
  RHS range        [1e+02, 1e+02]

MIP start from previous solve produced solution with objective 2450 (0.01s)
Loaded MIP start from previous solve with objective 2450

Presolve removed 3 rows and 3 columns
Presolve time: 0.00s
Presolved: 4 rows, 6 columns, 9 nonzeros
Variable types: 0 continuous, 6 integer (3 binary)

Root relaxation: objective 2.575000e+03, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |

**Exercise 1 (Solution)**

Result: C is still not made. The plan stays A = 40, B = 40 and the profit stays 2450.

Why: Setup cost is paid once, but labor is the scarce resource. C earns 50/3 ≈ 16.7 per labor hour before setup, against 20 for A and 30 for B. Take 30 units of C (90 hours) plus 30 units of B (30 hours): 1500 + 900 − 50 − 150 = 2200, which is below 2450. A cheap setup cannot fix a poor return per hour.

Exercise 2: No setup costs. Set all three setup costs to 0. What is the new plan? How does it compare with the plan with setup costs?


In [ ]:
setup = {"A": 0, "B": 0, "C": 0}


In [ ]:
m.update()
m.setObjective(gp.quicksum(profit[p]*x[p] - setup[p]*y[p] for p in products), GRB.MAXIMIZE)
m.optimize()

if m.Status == GRB.OPTIMAL:
    print(f"Maximum net profit: {m.ObjVal:.2f}\n")
    for p in products:
        print(f"Product {p}:  make it? {y[p].X > 0.5}   units = {round(x[p].X)}   batches = {round(k[p].X)}")
else:
    print("The model was not solved to optimality. Status code:", m.Status)
print(f"\nNet profit with no setup cost: {m.ObjVal: .0f}")

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.5 LTS")

CPU model: AMD EPYC 7B12, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 7 rows, 9 columns and 15 nonzeros (Max)
Model fingerprint: 0xf4baf7c2
Model has 3 linear objective coefficients
Variable types: 3 continuous, 6 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+01]
  Objective range  [3e+01, 5e+01]
  Bounds range     [1e+00, 5e+00]
  RHS range        [1e+02, 1e+02]

MIP start from previous solve produced solution with objective 2800 (0.01s)
Loaded MIP start from previous solve with objective 2800

Presolve removed 7 rows and 9 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 2 available processors)

Solution count 1: 2800 

Optimal solution found (tolerance 1.00e-04)
Best objecti

**Exercise 2 (Solution)**

Result: Plan is A = 40, B = 40, C = 0, with profit 2800.

Why: This is the same plan as before, but the 350 of setup cost disappears (2450 + 350 = 2800). C is still unprofitable per labor hour, so it is still not made. Gurobi may show y[C] = 1 with 0 units. That is harmless, since a setup that costs 0 makes no difference to the objective.

Exercise 3: Smaller batches. Set batch_size = 1 and max_batches = 60. The factory can now produce any whole number of units. Does the net profit go up, down or stay the same? Why?

In [ ]:
# 1. Redefine the original data with smaller batch size
products = ["A", "B", "C"]
profit   = {"A": 40,  "B": 30,  "C": 50}
setup    = {"A": 200, "B": 150, "C": 300}  # back to original setup costs
hours    = {"A": 2,   "B": 1,   "C": 3}
maxunits = {"A": 40,  "B": 60,  "C": 30}

labor_available = 120
batch_size      = 1   # Exercise 3: Smaller batch size
max_batches     = 60  # Exercise 3: Max batches

# 2. Build the new model
m3 = gp.Model("exercise_3")

# Add variables
x3 = m3.addVars(products, lb=0, vtype=GRB.CONTINUOUS, name="x")
y3 = m3.addVars(products, vtype=GRB.BINARY, name="y")
k3 = m3.addVars(products, lb=0, ub=max_batches, vtype=GRB.INTEGER, name="k")

# Set Objective
m3.setObjective(gp.quicksum(profit[p]*x3[p] - setup[p]*y3[p] for p in products), GRB.MAXIMIZE)

# Add Constraints
m3.addConstrs((x3[p] <= maxunits[p]*y3[p] for p in products), name="link")
m3.addConstr(gp.quicksum(hours[p]*x3[p] for p in products) <= labor_available, name="labor")
m3.addConstrs((x3[p] == batch_size*k3[p] for p in products), name="batch")

# Solve
m3.update()
m3.optimize()


Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.5 LTS")

CPU model: AMD EPYC 7B12, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 7 rows, 9 columns and 15 nonzeros (Max)
Model fingerprint: 0x7875b2bf
Model has 6 linear objective coefficients
Variable types: 3 continuous, 6 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+01]
  Objective range  [3e+01, 3e+02]
  Bounds range     [1e+00, 6e+01]
  RHS range        [1e+02, 1e+02]

Found heuristic solution: objective -0.0000000
Presolve removed 3 rows and 3 columns
Presolve time: 0.00s
Presolved: 4 rows, 6 columns, 9 nonzeros
Variable types: 0 continuous, 6 integer (3 binary)

Root relaxation: objective 2.700000e+03, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time


In [ ]:
if m.Status == GRB.OPTIMAL:
    print(f"Maximum net profit: {m3.ObjVal:.2f}\n")
    for p in products:
        print(f"Product {p}:  make it? {y3[p].X > 0.5}   units = {round(x3[p].X)}   batches = {round(k3[p].X)}")
else:
    print("The model was not solved to optimality. Status code:", m3.Status)

Maximum net profit: 2650.00

Product A:  make it? True   units = 30   batches = 30
Product B:  make it? True   units = 60   batches = 60
Product C:  make it? False   units = 0   batches = 0


**Exercise 3 (Solution)**

Result: Plan is A = 30, B = 60, C = 0, with profit 2650. Profit goes up by 200.

Working:

Labor used is 2·30 + 60 = 120 hours.
Profit is 40·30 + 30·60 − 200 − 150 = 1200 + 1800 − 350 = 2650.

Why: Removing the batch rule only enlarges the set of feasible plans, so profit cannot go down. Here it also unlocks B’s full market limit of 60, which was out of reach with the 5-batch cap. B gives the most profit per hour (30), so it should get the labor first.

Exercise 4: A new rule. Add a constraint so that the factory makes at least 50 units of product B, and solve. What is the new plan and net profit? Why did it get worse? Hint: a constraint on one variable looks like m.addConstr(x["B"] >= 50, name="min_B").

In [ ]:
# set setup back to the original values and rebuild the objective once, so later exercises start from the original model
setup = {"A": 200, "B": 150, "C": 300}
m.setObjective(gp.quicksum(profit[p]*x[p] - setup[p]*y[p] for p in products), GRB.MAXIMIZE)


In [ ]:
m.addConstr(x["B"] >= 50, name="min_B")
m.optimize()
print(f"Net profit: {m.ObjVal:.0f}")
for p in products:
    print(f"Product {p}: units = {round(x[p].X)}")

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.5 LTS")

CPU model: AMD EPYC 7B12, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 8 rows, 9 columns and 16 nonzeros (Max)
Model fingerprint: 0x09e2ae2d
Model has 6 linear objective coefficients
Variable types: 3 continuous, 6 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+01]
  Objective range  [3e+01, 3e+02]
  Bounds range     [1e+00, 5e+00]
  RHS range        [5e+01, 1e+02]

MIP start from previous solve did not produce a new incumbent solution

Presolve removed 5 rows and 5 columns
Presolve time: 0.00s
Presolved: 3 rows, 4 columns, 6 nonzeros
Variable types: 0 continuous, 4 integer (2 binary)
Found heuristic solution: objective 1350.0000000
Found heuristic solution: objective 1550.0000000
Found heuristic solution: objective 2050.0000000

Root relaxation: objective 2.466667e+03, 1 iterations, 0.00 secon

**Exercise 4 (Solution)**

Result: Plan is A = 30, B = 50, C = 0, with profit 2350. This is 100 worse than 2450.

Working:

B must be at least 50, and the batch cap makes that exactly 50.
B uses 50 hours, so 70 hours are left for A. That allows 35 units, rounded down to 30 because of batches.
Profit is 1200 + 1500 − 350 = 2350.

Why: Compared with the baseline, the plan gives up 10 units of A (−400) and gains 10 units of B (+300), a net loss of 100. A constraint that cuts off the old optimum can only make the objective worse or leave it unchanged.

Exercise 5: Your own product. Add a fourth product D by editing only the data cell: products, profit, setup, hours and maxunits each need one new entry. Notice that you do not need to change any model code. Why is that?

In [ ]:
m.remove(m.getConstrByName("min_B"))
m.optimize()
print(m.ObjVal)   # should be back to 2450

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.5 LTS")

CPU model: AMD EPYC 7B12, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 7 rows, 9 columns and 15 nonzeros (Max)
Model fingerprint: 0x90cfe33c
Model has 6 linear objective coefficients
Variable types: 3 continuous, 6 integer (3 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+01]
  Objective range  [3e+01, 3e+02]
  Bounds range     [1e+00, 5e+00]
  RHS range        [1e+02, 1e+02]

MIP start from previous solve produced solution with objective 2350 (0.01s)
Loaded MIP start from previous solve with objective 2350

Presolve removed 3 rows and 3 columns
Presolve time: 0.00s
Presolved: 4 rows, 6 columns, 9 nonzeros
Variable types: 0 continuous, 6 integer (3 binary)

Root relaxation: objective 2.575000e+03, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |

In [ ]:
# Separate data for the 4-product version (originals stay untouched)
products5 = ["A", "B", "C", "D"]
profit5   = {"A": 40,  "B": 30,  "C": 50,  "D": 45}
setup5    = {"A": 200, "B": 150, "C": 300, "D": 100}
hours5    = {"A": 2,   "B": 1,   "C": 3,   "D": 2}
maxunits5 = {"A": 40,  "B": 60,  "C": 30,  "D": 40}

# Same model code, only the names changed
m5 = gp.Model("exercise_5")
x5 = m5.addVars(products5, lb=0, vtype=GRB.CONTINUOUS, name="x")
y5 = m5.addVars(products5, vtype=GRB.BINARY, name="y")
k5 = m5.addVars(products5, lb=0, ub=5, vtype=GRB.INTEGER, name="k")

m5.setObjective(gp.quicksum(profit5[p]*x5[p] - setup5[p]*y5[p] for p in products5), GRB.MAXIMIZE)
m5.addConstrs((x5[p] <= maxunits5[p]*y5[p] for p in products5), name="link")
m5.addConstr(gp.quicksum(hours5[p]*x5[p] for p in products5) <= 120, name="labor")
m5.addConstrs((x5[p] == 10*k5[p] for p in products5), name="batch")

m5.optimize()

print(f"Net profit: {m5.ObjVal:.0f}")
for p in products5:
    print(f"Product {p}: make it? {y5[p].X > 0.5}   units = {round(x5[p].X)}")

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 24.04.5 LTS")

CPU model: AMD EPYC 7B12, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 9 rows, 12 columns and 20 nonzeros (Max)
Model fingerprint: 0x8f6bc656
Model has 8 linear objective coefficients
Variable types: 4 continuous, 8 integer (4 binary)
Coefficient statistics:
  Matrix range     [1e+00, 6e+01]
  Objective range  [3e+01, 3e+02]
  Bounds range     [1e+00, 5e+00]
  RHS range        [1e+02, 1e+02]

Found heuristic solution: objective -0.0000000
Presolve removed 4 rows and 4 columns
Presolve time: 0.00s
Presolved: 5 rows, 8 columns, 12 nonzeros
Variable types: 0 continuous, 8 integer (4 binary)

Root relaxation: objective 2.837500e+03, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Tim

**Exercise 5 (Solution)**

I chose my own numbers for D: profit 45, setup 100, 2 hours per unit, market limit 40. I only changed products, profit, setup, hours and maxunits in the data cell and nothing else.

Result: D is made and the plan changes to B = 40 units, D = 40 units, A = 0 and C = 0, with a net profit of 2750 (up from 2450). D earns 45/2 = 22.5 per labor hour, which is above A (20) and below only B (30), and its setup cost (100) is lower than A's (200). So D replaces A in the plan. Check: labor used = 40 + 80 = 120 hours, and profit = 1200 + 1800 - 150 - 100 = 2750.

Why no model code changes:

The model code does not need to change because every variable, objective term and constraint is written as a loop over the list products, and all numbers come from dictionaries keyed by product name. Adding D to the list and to each dictionary is enough for addVars, quicksum and addConstrs to create the extra variables, objective term and constraints automatically. Separating data from the model is what makes it reusable.

Exercise 6: Counting constraints. After you add product D, print m.NumVars and m.NumConstrs. Predict the numbers before you run the cell.

In [ ]:
m5.update()
print("Variables  :", m5.NumVars)
print("Constraints:", m5.NumConstrs)

Variables  : 12
Constraints: 9


**Exercise 6 (Solution)**

Counting variables and constraints with D

Variables: 3 types (x, y, k) × 4 products = 12.

Constraints:
link: 4
batch: 4
labor: 1
Total: 9

These counts are the same whatever numbers you choose for D. They depend only on the number of products.

The original model had 3 × 3 = 9 variables and 3 + 3 + 1 = 7 constraints.